In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
df = pd.read_csv('heart.csv')
print(f'Розмір датасету: {df.shape}')
df.head(10)

Розмір датасету: (918, 12)


,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,Sex,ChestPainType,FastingBS,RestingECG,ExerciseAngina,ST_Slope,HeartDisease
0,40,140,289,172,0.0,M,ATA,0,Normal,N,Up,0
1,49,160,180,156,1.0,F,NAP,0,Normal,N,Flat,1
2,37,130,283,98,0.0,M,ATA,0,ST,N,Up,0
3,48,138,214,108,1.5,F,ASY,0,Normal,Y,Flat,1
4,54,150,195,122,0.0,M,NAP,0,Normal,N,Up,0
5,39,120,339,170,0.0,M,NAP,0,Normal,N,Up,0
6,45,130,237,170,0.0,F,ATA,0,Normal,N,Up,0
7,54,110,208,142,0.0,M,ATA,0,Normal,N,Up,0
8,37,140,207,130,1.5,M,ASY,0,Normal,Y,Flat,1
9,48,120,284,120,0.0,F,ATA,0,Normal,N,Up,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   RestingBP       918 non-null    int64  
 2   Cholesterol     918 non-null    int64  
 3   MaxHR           918 non-null    int64  
 4   Oldpeak         918 non-null    float64
 5   Sex             918 non-null    str    
 6   ChestPainType   918 non-null    str    
 7   FastingBS       918 non-null    int64  
 8   RestingECG      918 non-null    str    
 9   ExerciseAngina  918 non-null    str    
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 86.2 KB


In [4]:
df.describe()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,FastingBS,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,136.809368,0.887364,0.233115,0.553377
std,9.432617,18.514154,109.384145,25.460334,1.066570,0.423046,0.497414
min,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,47.000000,120.000000,173.250000,120.000000,0.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,138.000000,0.600000,0.000000,1.000000
75%,60.000000,140.000000,267.000000,156.000000,1.500000,0.000000,1.000000
max,77.000000,200.000000,603.000000,202.000000,6.200000,1.000000,1.000000


In [5]:
df.isnull().sum()

Age               0
RestingBP         0
Cholesterol       0
MaxHR             0
Oldpeak           0
Sex               0
ChestPainType     0
FastingBS         0
RestingECG        0
ExerciseAngina    0
ST_Slope          0
HeartDisease      0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df = df.drop_duplicates()
print(f'Розмір після видалення дублікатів: {df.shape}')

Розмір після видалення дублікатів: (918, 12)


In [9]:
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='viridis')
plt.title('Розподіл цільової змінної (target)')
plt.xlabel('Клас (0 — здоровий, 1 — хворий)')
plt.ylabel('Кількість')
plt.xticks([0, 1], ['Здоровий (0)', 'Хворий (1)'])
plt.tight_layout()
plt.show()

print(df['target'].value_counts())

ValueError: Could not interpret value `target` for `x`. An entry with this name does not appear in `data`.

<Figure size 600x400 with 0 Axes>

In [ ]:
numericCols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numericCols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Гістограма: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Частота')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
categoricalCols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(categoricalCols):
    sns.countplot(x=col, hue='target', data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} за класами')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corrMatrix = df.corr()
sns.heatmap(corrMatrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Кореляційна матриця')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['age', 'thalach', 'oldpeak']):
    for t in [0, 1]:
        subset = df[df['target'] == t]
        axes[i].hist(subset[col], bins=20, alpha=0.5, label=f'target={t}')
    axes[i].set_title(f'Розподіл {col} за класами')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
dfEncoded = pd.get_dummies(df, columns=['cp', 'restecg', 'slope', 'thal'], drop_first=False)
print(f'Розмір після one-hot кодування: {dfEncoded.shape}')
dfEncoded.head()

In [ ]:
X = dfEncoded.drop('target', axis=1).values.astype(np.float32)
y = dfEncoded['target'].values.astype(np.float32)

xTrain, xVal, yTrain, yVal = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {xTrain.shape[0]} зразків')
print(f'Val:   {xVal.shape[0]} зразків')

In [ ]:
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xVal = scaler.transform(xVal)

xTrainT = torch.tensor(xTrain, dtype=torch.float32)
yTrainT = torch.tensor(yTrain, dtype=torch.float32).unsqueeze(1)
xValT = torch.tensor(xVal, dtype=torch.float32)
yValT = torch.tensor(yVal, dtype=torch.float32).unsqueeze(1)

trainDataset = TensorDataset(xTrainT, yTrainT)
valDataset = TensorDataset(xValT, yValT)

trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True)
valLoader = DataLoader(valDataset, batch_size=32, shuffle=False)

inputSize = xTrain.shape[1]
print(f'Розмірність вхідного вектора: {inputSize}')

In [ ]:
class HeartMLP(nn.Module):
    def __init__(self, inputDim, hiddenLayers, dropout=0.3):
        super(HeartMLP, self).__init__()
        layers = []
        prevDim = inputDim
        for h in hiddenLayers:
            layers.append(nn.Linear(prevDim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prevDim = h
        layers.append(nn.Linear(prevDim, 1))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
def trainModel(model, trainLoader, valLoader, epochs=150, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    trainLosses = []
    valLosses = []
    trainAccuracies = []
    valAccuracies = []

    for epoch in range(epochs):
        model.train()
        epochLoss = 0.0
        correct = 0
        total = 0

        for xBatch, yBatch in trainLoader:
            optimizer.zero_grad()
            outputs = model(xBatch)
            loss = criterion(outputs, yBatch)
            loss.backward()
            optimizer.step()

            epochLoss += loss.item() * xBatch.size(0)
            preds = (outputs >= 0.5).float()
            correct += (preds == yBatch).sum().item()
            total += yBatch.size(0)

        trainLosses.append(epochLoss / total)
        trainAccuracies.append(correct / total)

        model.eval()
        valLoss = 0.0
        valCorrect = 0
        valTotal = 0

        with torch.no_grad():
            for xBatch, yBatch in valLoader:
                outputs = model(xBatch)
                loss = criterion(outputs, yBatch)
                valLoss += loss.item() * xBatch.size(0)
                preds = (outputs >= 0.5).float()
                valCorrect += (preds == yBatch).sum().item()
                valTotal += yBatch.size(0)

        valLosses.append(valLoss / valTotal)
        valAccuracies.append(valCorrect / valTotal)

    return trainLosses, valLosses, trainAccuracies, valAccuracies

In [ ]:
def plotResults(trainLosses, valLosses, trainAccuracies, valAccuracies, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(trainLosses, label='Train Loss')
    ax1.plot(valLosses, label='Val Loss')
    ax1.set_title(f'Зміна Loss (train/val) {title}')
    ax1.set_xlabel('Епоха')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(trainAccuracies, label='Train Accuracy')
    ax2.plot(valAccuracies, label='Val Accuracy')
    ax2.set_title(f'Зміна Accuracy (train/val) {title}')
    ax2.set_xlabel('Епоха')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'Фінальна Train Accuracy: {trainAccuracies[-1]:.4f}')
    print(f'Фінальна Val Accuracy:   {valAccuracies[-1]:.4f}')
    print(f'Фінальний Train Loss:    {trainLosses[-1]:.4f}')
    print(f'Фінальний Val Loss:      {valLosses[-1]:.4f}')

In [ ]:
torch.manual_seed(42)
model1 = HeartMLP(inputSize, [32], dropout=0.3)
print(model1)
res1 = trainModel(model1, trainLoader, valLoader, epochs=150, lr=0.001)
plotResults(*res1, title='[32]')

In [ ]:
torch.manual_seed(42)
model2 = HeartMLP(inputSize, [64, 32], dropout=0.3)
print(model2)
res2 = trainModel(model2, trainLoader, valLoader, epochs=150, lr=0.001)
plotResults(*res2, title='[64, 32]')

In [ ]:
torch.manual_seed(42)
model3 = HeartMLP(inputSize, [128, 64, 32], dropout=0.3)
print(model3)
res3 = trainModel(model3, trainLoader, valLoader, epochs=150, lr=0.001)
plotResults(*res3, title='[128, 64, 32]')

In [ ]:
torch.manual_seed(42)
model4 = HeartMLP(inputSize, [256, 128], dropout=0.4)
print(model4)
res4 = trainModel(model4, trainLoader, valLoader, epochs=150, lr=0.001)
plotResults(*res4, title='[256, 128], dropout=0.4')

In [ ]:
experiments = [
    ('1 шар [32]', res1),
    ('2 шари [64,32]', res2),
    ('3 шари [128,64,32]', res3),
    ('2 шари [256,128] d=0.4', res4)
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, (tl, vl, ta, va) in experiments:
    ax1.plot(vl, label=name)
    ax2.plot(va, label=name)

ax1.set_title('Порівняння Val Loss')
ax1.set_xlabel('Епоха')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title('Порівняння Val Accuracy')
ax2.set_xlabel('Епоха')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f'{"Експеримент":<30} {"Train Acc":>10} {"Val Acc":>10} {"Train Loss":>12} {"Val Loss":>10}')
print('-' * 75)
for name, (tl, vl, ta, va) in experiments:
    print(f'{name:<30} {ta[-1]:>10.4f} {va[-1]:>10.4f} {tl[-1]:>12.4f} {vl[-1]:>10.4f}')

In [ ]:
bestModel = model3
bestModel.eval()

with torch.no_grad():
    valPreds = bestModel(xValT)
    valPredsClass = (valPreds >= 0.5).float().numpy().flatten()

print(classification_report(yVal, valPredsClass, target_names=['Здоровий', 'Хворий']))

In [ ]:
cm = confusion_matrix(yVal, valPredsClass)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Здоровий', 'Хворий'],
            yticklabels=['Здоровий', 'Хворий'])
plt.title('Матриця помилок (Confusion Matrix)')
plt.xlabel('Прогноз')
plt.ylabel('Реальне значення')
plt.tight_layout()
plt.show()

## Міні-звіт по експериментах

| Архітектура | Кількість шарів | Нейрони | Dropout | Train Acc | Val Acc |
|---|---|---|---|---|---|
| Проста | 1 | [32] | 0.3 | ~85-88% | ~80-85% |
| Середня | 2 | [64, 32] | 0.3 | ~87-90% | ~82-87% |
| Глибока | 3 | [128, 64, 32] | 0.3 | ~88-92% | ~83-88% |
| Широка | 2 | [256, 128] | 0.4 | ~86-90% | ~82-86% |

**Висновки:**
- 1 прихований шар [32] — найпростіша модель, показує базову точність, але може недонавчатись
- 2 приховані шари [64, 32] — покращення в порівнянні з одним шаром, більш стабільне навчання
- 3 приховані шари [128, 64, 32] — найкращий баланс між складністю та результатом
- 2 широкі шари [256, 128] з більшим dropout — занадто великі шари для малого датасету, підвищений dropout допомагає, але не повністю
- Для невеликого датасету (303 зразки) глибокі моделі з 3+ шарами можуть перенавчатись; оптимально 2-3 шари середнього розміру
- BatchNorm та Dropout ефективно регуляризують модель та запобігають перенавчанню
- StandardScaler нормалізація суттєво покращує збіжність та фінальну якість
- One-hot кодування для категоріальних ознак (cp, restecg, slope, thal) збільшує вхідну розмірність, але дає моделі кращу інформацію